# InferenceGuard: Model Training & Multi-Dimensional Evaluation Pipeline

**Objective**: End-to-end training (Position-Controlled SFT + Contrastive DPO Alignment) and multi-dimensional evaluation of the privacy-preserving InferenceGuard Rewriter.

### Pipeline Architecture
1. **Externalized Persona Banks**: 210+ cities, 156 jobs, 105 universities, age/income patterns (`data/rewriter/personas/`).
2. **Position-Controlled Augmentation**: Deterministic, balanced synthetic corpus across preamble, postamble, mid-sentence, parenthetical, and split positions (`src/rewriter/augment.py`).
3. **Decompose-and-Reconstruct Policy**: Structural prompt builder (`src/rewriter/prompting.py`) ensuring 0% train/serve drift and downstream task preservation.
4. **Supervised Fine-Tuning (SFT)**: QLoRA 4-bit fine-tuning of Qwen student model (`src/rewriter/train_sft.py`).
5. **Direct Preference Optimization (DPO)**: Contrastive alignment penalizing over-scrubbing (truncation) and under-scrubbing (PII leaks) (`src/rewriter/train_dpo.py`).
6. **Span-Annotated Benchmark**: 53-case evaluation with statutory invariance and compound request verification (`scripts/evaluate_benchmark_50.py`).
7. **Live Interactive Testing & Web Dashboard**: Real-time prompt rewriting and FastAPI UI.

## 1. Environment Setup & Repository Verification
Ensure the repository is cloned, navigate to the root directory, and verify core files.

In [ ]:
# Cell 1: Environment Setup & Repository Verification
import os
import sys
from pathlib import Path

# Verify or clone the repository
repo_dir = Path.cwd()
if not (repo_dir / 'src' / 'rewriter' / 'train_sft.py').exists():
    if Path('/content/Inference-Guard').exists():
        os.chdir('/content/Inference-Guard')
    elif Path('/content/InferenceGuard').exists():
        os.chdir('/content/InferenceGuard')
    else:
        print('Cloning repository...')
        !git clone -b feature/rewriter-teacher-distillation https://github.com/GIND123/Inference-Guard.git /content/Inference-Guard
        os.chdir('/content/Inference-Guard')

print('Active working directory:', Path.cwd())
assert (Path.cwd() / 'src' / 'rewriter' / 'train_sft.py').exists(), 'Repository root verification failed!'
print('Repository verified.')


## 2. Mount Google Drive for Fault-Tolerant Checkpoint Persistence
If running in Google Colab, mount Google Drive so intermediate training steps, final LoRA adapters, and evaluation reports are safely persisted.

In [ ]:
# Cell 2: Mount Google Drive (with graceful local fallback)
from pathlib import Path

drive_mounted = False
drive_base_dir = Path('/content/drive/MyDrive/InferenceGuard')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_mounted = True
    drive_base_dir.mkdir(parents=True, exist_ok=True)
    (drive_base_dir / 'rewriter_sft_0.5b').mkdir(parents=True, exist_ok=True)
    (drive_base_dir / 'rewriter_dpo_0.5b').mkdir(parents=True, exist_ok=True)
    print(f'Google Drive successfully mounted at {drive_base_dir}')
except Exception as exc:
    print(f'Google Drive not mounted (running locally or without Colab auth): {exc}')
    print('Checkpoints and outputs will be saved locally in artifacts/')


## 3. Install Dependencies & Inspect GPU Environment
Install the required ML libraries (`transformers`, `peft`, `trl`, `datasets`, `bitsandbytes`, `accelerate`) and prevent Colab C++ binary mismatches.

In [ ]:
# Cell 3: Install dependencies & inspect GPU
# Proactively remove conflicting multimodal extensions in Colab
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true

# Install required core packages
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate
!pip install -q fastapi uvicorn presidio-analyzer presidio-anonymizer pytest

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('Operating in CPU/MPS mode.')


## 4. Data Preparation: Persona Banks, Augmentation & Quality Gates
Inspect the externalized persona banks, generate a position-controlled augmented dataset, verify prompt parity (0% drift), and generate DPO preference pairs.

In [ ]:
# Cell 4: Inspect externalized persona banks
import json
from pathlib import Path
from src.rewriter.persona_banks import default_matcher

matcher = default_matcher()
print(f'Loaded {matcher.value_pattern_count} compiled patterns from data/rewriter/personas/')

sample_text = 'Living in Berlin, working as a DevOps engineer, studied at ETH Zurich, take the U-Bahn daily.'
matches = matcher.scan(sample_text)
print(f'\nPersona detection test on: "{sample_text}"')
for m in matches:
    print(f' - [{m.attribute.upper()}] "{m.surface}" -> generalizations: {m.generalizations}')


In [ ]:
# Cell 5: Generate position-balanced augmented dataset & verify parity gate
# 1. Compose 2,000 position-balanced synthetic rows (deterministic, no GPU required)
!python -m src.rewriter.augment --n 2000 --out data/sft/augmented_v1/train.jsonl

# 2. Enforce strict prompt parity and zero duplicate turns regression gate (must exit 0)
!python scripts/check_prompt_parity.py data/sft/augmented_v1/train.jsonl


In [ ]:
# Cell 6: Generate and inspect DPO preference pairs
# Generate contrastive preference pairs (prompt, chosen, rejected)
!python -m src.rewriter.generate_dpo_data --n 1500 --seed 42 --out data/dpo/train_dpo.jsonl

# Inspect sample preference pair
with open('data/dpo/train_dpo.jsonl', 'r', encoding='utf-8') as f:
    sample_dpo = json.loads(f.readline())

print('=== DPO Pair Sample ===')
print(f'Failure Mode Target: {sample_dpo["metadata"]["failure_mode"]}')
print(f'Prompt:   {sample_dpo["prompt"][-1]["content"]}')
print(f'CHOSEN:   {sample_dpo["chosen"]}')
print(f'REJECTED: {sample_dpo["rejected"]}')


## 5. Stage 1: Supervised Fine-Tuning (SFT) with QLoRA
Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` using 4-bit QLoRA and completion-only loss masking on the position-balanced augmented dataset.

In [ ]:
# Cell 7: Fine-tune Qwen 0.5B Student Model (SFT)
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true
from pathlib import Path

drive_flag = (
    '--drive_path /content/drive/MyDrive/InferenceGuard/rewriter_sft_0.5b'
    if Path('/content/drive/MyDrive/InferenceGuard').exists()
    else ''
)

!python -m src.rewriter.train_sft \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --dataset_path data/sft/augmented_v1/train.jsonl \
    --output_adapter_dir artifacts/rewriter_qlora_0.5b \
    --epochs 3 \
    --batch_size 4 \
    --grad_accum 4 \
    --learning_rate 2e-4 \
    --lora_r 32 \
    --lora_alpha 64 \
    --warmup_ratio 0.05 \
    {drive_flag}


## 6. Stage 2: Direct Preference Optimization (DPO) Alignment
Align the student model using DPO to penalize over-scrubbing (cutting off downstream questions) and under-scrubbing (leaving PII cues).

In [ ]:
# Cell 8: Direct Preference Optimization (DPO)
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true

!python -m src.rewriter.train_dpo \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --ref_adapter_path artifacts/rewriter_qlora_0.5b \
    --dataset_path data/dpo/train_dpo.jsonl \
    --output_dir artifacts/rewriter_dpo_0.5b \
    --epochs 1 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 5e-6 \
    --beta 0.1 \
    --max_length 512 \
    --max_prompt_length 384


## 7. Model Promotion to Production
Promote the best student model (`artifacts/rewriter_dpo_0.5b` or `artifacts/rewriter_qlora_0.5b`) to the canonical production path `artifacts/rewriter_qlora`.

In [ ]:
# Cell 9: Promote Best Adapter to Production
import shutil
from pathlib import Path

source_dir = Path('artifacts/rewriter_dpo_0.5b')
if not (source_dir / 'adapter_model.safetensors').exists():
    source_dir = Path('artifacts/rewriter_qlora_0.5b')

prod_dir = Path('artifacts/rewriter_qlora')
prod_dir.mkdir(parents=True, exist_ok=True)

for item in source_dir.glob('*'):
    if item.is_file():
        shutil.copy2(item, prod_dir / item.name)

print(f'Promoted adapter from {source_dir} to {prod_dir}:')
for f in sorted(prod_dir.glob('*')):
    if f.is_file():
        print(f' - {f.name} ({f.stat().st_size / 1024:.1f} KB)')


## 8. Comprehensive Span-Annotated Benchmark Evaluation (53 Cases)
Evaluate the promoted model across all 53 span-annotated test cases, assessing PII span removal, invariant preservation, copy-through, and compound request handling.

In [ ]:
# Cell 10: Run Span-Annotated Benchmark Suite on Trained Model
!PYTHONPATH=. python scripts/evaluate_benchmark_50.py \
    --adapter_path artifacts/rewriter_qlora \
    --benchmark_file data/evaluation/benchmark_test_50.jsonl \
    --output_report reports/benchmark_50_results.json


In [ ]:
# Cell 11: Benchmark Fallback Baseline (The Mechanical Floor)
!PYTHONPATH=. python scripts/evaluate_benchmark_50.py \
    --fallback_baseline \
    --benchmark_file data/evaluation/benchmark_test_50.jsonl \
    --output_report reports/benchmark_50_fallback_results.json


## 9. Run Full Regression & Integration Test Suite
Execute the Pytest test suite covering the generalization fix, e2e inference, and fallbacks.

In [ ]:
# Cell 12: Run Regression Test Suite
!pytest tests/test_rewriter_generalization_fix.py tests/test_e2e_qwen_inference.py tests/test_stress_loader_fallback.py -v


## 10. Interactive Privacy Rewriter Demonstration
Run live inference on complex user prompts containing preambles, postambles, mid-sentence personal disclosures, third-party invariants, and embedded compound requests.

In [ ]:
# Cell 13: Live Interactive Inference Demonstration
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true

import sys
import torch
from pathlib import Path
from src.rewriter.inference import QwenRewriterInference

rewriter = QwenRewriterInference(
    model_name_or_path='Qwen/Qwen2.5-0.5B-Instruct',
    adapter_path='artifacts/rewriter_qlora',
)

test_scenarios = [
    # 1. Co-occurring PII preamble
    'I am a 34-year-old senior oncologist at Johns Hopkins in Baltimore earning $320k. What is the standard adjuvant protocol for HER2-positive stage II breast cancer?',
    # 2. Trailing postamble PII
    'What is the recommended antibiotic regimen for acute bacterial meningitis? FYI, I am a 29-year-old internal medicine resident in Chicago working nights.',
    # 3. Mid-sentence embedded demographic
    'How do I configure a distributed Ray cluster on AWS as a 26-year-old engineer in Austin making $110k with low latency?',
    # 4. Invariant legal statute & third-party subject (MUST BE PRESERVED)
    'Under Tokyo Metropolitan Building Ordinance, what are seismic bracing rules for a 5-year-old pediatric patient clinic? Note: I am an architect in Osaka.',
    # 5. Embedded compound request (MUST NOT TRUNCATE DOWNSTREAM REQUESTS)
    'How do I overcome stage fright as a 42 year old software professional, help me find resources for public speaking',
]

print(f'Rewriter Loaded (Adapter: {rewriter.is_adapter_loaded}):\n' + '=' * 80)
for idx, prompt in enumerate(test_scenarios, 1):
    output = rewriter.rewrite(prompt)
    print(f'\n[Test Case {idx}]:')
    print(f'INPUT:     {prompt}')
    print(f'REWRITTEN: {output}')
    print('-' * 80)


## 11. Launch InferenceGuard Interactive Web UI
Launch the FastAPI + Uvicorn server in the background and expose the dashboard.

In [ ]:
# Cell 14: Launch InferenceGuard Web UI (Uvicorn + Colab Port Forwarding / Tunnel)
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pkill -f uvicorn 2>/dev/null || true

import subprocess
import time
from pathlib import Path

# Launch Uvicorn server in background
proc = subprocess.Popen(
    ['uvicorn', 'web.api:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(3)

print('InferenceGuard Web Server running at http://localhost:8000')
print('Swagger OpenAPI docs available at http://localhost:8000/docs')

# Optional: Connect via pyngrok in Colab
# from pyngrok import ngrok
# public_url = ngrok.connect(8000)
# print(f'Public Web URL: {public_url}')
